# HỆ THỐNG QUẢN LÝ HOÀN TIỀN VÀ ĐIỂM THƯỞNG

**Học phần:** Cấu trúc dữ liệu tài chính và giải thuật  
**Môi trường:** Google Colab hoặc Jupyter Notebook · Python 3  
**Thành viên:** cập nhật tên thật trước khi nộp

> Đây là mô hình học thuật sử dụng dữ liệu giả lập, không kết nối ngân hàng và không xử lý tiền thật. Tỷ lệ thưởng bên dưới chỉ để minh họa.

## Cách chạy

- **Google Colab:** tải tệp `.ipynb` này lên Google Drive, nhấp phải **Open with → Google Colaboratory**, sau đó chọn **Runtime → Run all**.
- **Jupyter Notebook/JupyterLab:** mở tệp này từ Jupyter và chọn **Run All Cells**.
- Không cần cài thêm thư viện hay upload tệp dữ liệu. Notebook tự tạo dữ liệu giả lập và giữ trạng thái trong bộ nhớ của kernel. Chạy lại cell tạo `book` để bắt đầu phiên demo mới.

## Mục tiêu

Minh họa sổ giao dịch, tra cứu mã, sắp xếp, duyệt hàng chờ FIFO hoặc ưu tiên, đổi/thu hồi điểm. Các cấu trúc theo bảng môn học: List, Queue, Stack, Hash Table và Heap. Trong Python, Queue dùng `collections.deque`, Stack/List dùng `list`, Hash Table dùng `dict`, và Heap dùng `heapq`. Merge sort là thuật toán sắp xếp, không phải cấu trúc dữ liệu.

## 1. Quy tắc nghiệp vụ

| Quy tắc giả lập | Giá trị |
| --- | ---: |
| Hoàn tiền cho giao dịch đã duyệt, làm tròn xuống | 1% |
| Tích điểm, lấy phần nguyên | 1 điểm / 10.000 ₫ |
| Giao dịch cần duyệt thêm | Trên 5.000.000 ₫ |
| Đổi điểm | Tối thiểu 100, bội số của 100 |
| Giá trị đổi | 1 điểm = 100 ₫ |

Một giao dịch đang chờ chưa tạo số dư thưởng. Nếu hội viên đã sử dụng điểm phát sinh từ giao dịch, yêu cầu hoàn bị từ chối để tránh số dư âm.

In [ ]:
from collections import deque
from copy import deepcopy
import heapq
import random
from statistics import fmean
from time import perf_counter

NGUONG_DUYET_VND = 5_000_000
VND_MOI_DIEM = 10_000
VND_DOI_MOI_DIEM = 100
BUOC_DOI_DIEM = 100

class LoiNghiepVu(ValueError):
    """Lỗi dữ liệu đầu vào hoặc vi phạm quy tắc thưởng."""

def merge_sort_giao_dich(giao_dich, thu_tu="desc"):
    """Sắp xếp merge sort O(n log n), dùng mã làm khóa phụ ổn định."""
    if thu_tu not in ("asc", "desc"):
        raise LoiNghiepVu("Thứ tự sắp xếp phải là asc hoặc desc.")

    def khoa(gd):
        tien = gd["so_tien_vnd"]
        return (tien if thu_tu == "asc" else -tien, gd["ma_gd"])

    def sap_xep(ds):
        if len(ds) < 2:
            return ds
        giua = len(ds) // 2
        trai, phai = sap_xep(ds[:giua]), sap_xep(ds[giua:])
        ket_qua, i, j = [], 0, 0
        while i < len(trai) and j < len(phai):
            if khoa(trai[i]) <= khoa(phai[j]):
                ket_qua.append(trai[i]); i += 1
            else:
                ket_qua.append(phai[j]); j += 1
        return ket_qua + trai[i:] + phai[j:]

    return sap_xep(list(giao_dich))

print("Đã nạp cấu trúc dữ liệu, hằng số nghiệp vụ và merge sort.")

## 2. Cấu trúc dữ liệu và phân tích thuật toán

- **List:** lưu danh sách giao dịch; cũng dùng làm Stack lưu snapshot undo.
- **Queue:** `collections.deque` lấy giao dịch đến trước bằng `popleft()` trong O(1).
- **Stack:** dùng `list`, lưu trạng thái trước thao tác; lấy phần tử gần nhất để undo.
- **Hash Table:** `dict` lập chỉ mục mã giao dịch/hội viên, tra cứu trung bình O(1), xấu nhất O(n).
- **Heap:** `heapq` lưu khóa `(-số tiền, mã)` để lấy giao dịch có số tiền cao nhất.
- Merge sort là thuật toán sắp xếp trên List, không phải cấu trúc dữ liệu riêng.

| Thao tác | Độ phức tạp | Lý do |
| --- | --- | --- |
| Tìm mã bằng Hash Table | O(1) trung bình; O(n) xấu nhất | Tính cả thời gian tính hash và va chạm |
| Merge sort n giao dịch | O(n log n), mọi trường hợp | Chia đôi theo log n mức; mỗi mức trộn O(n) |
| Lấy đầu Queue đã dựng | O(1) | `deque.popleft()` không dời các phần tử còn lại |
| Lấy phần tử từ Heap đã dựng | O(log q) | Sau khi lấy, Heap khôi phục tính chất qua các tầng |
| Một lần bấm duyệt hiện tại | O(n + q) | Mỗi lần quét danh sách n giao dịch, dựng Heap q phần tử rồi lấy ưu tiên |
| Push/pop Stack | O(1) | Chỉ thêm hoặc lấy snapshot ở đỉnh |

Trong đó n là tổng giao dịch, q là số giao dịch chờ. Code dựng lại Queue/Heap từ danh sách giao dịch ở mỗi lần gọi; vì thế toàn bộ lần duyệt FIFO hiện tại là O(n), còn duyệt ưu tiên là O(n + q). O(1) của Queue và O(log q) của Heap chỉ tính thao tác lấy phần tử sau khi cấu trúc đã được dựng.

### Mã giả merge sort

```text
MERGE_SORT(A):
  nếu độ dài A < 2: trả A
  chia A thành trái và phải
  trái = MERGE_SORT(trái); phải = MERGE_SORT(phải)
  trộn hai nửa theo (số tiền, mã giao dịch)
  trả danh sách đã trộn
```

### Mã giả duyệt ưu tiên

```text
DUYỆT_ƯU_TIÊN:
  tạo khóa (-số tiền, mã) cho từng giao dịch đang chờ
  HEAPIFY(danh sách khóa)
  lấy khóa nhỏ nhất (tương ứng số tiền lớn nhất)
  đánh dấu giao dịch đã duyệt
```

Ví dụ 6, 9, 7 triệu tạo khóa -6, -9, -7 triệu. Min-heap lấy -9 trước nên giao dịch 9 triệu được duyệt trước. Dấu âm chỉ đổi chiều ưu tiên, không đổi số tiền giao dịch.

In [ ]:
class SoThuong:
    """Ledger giả lập; dữ liệu sống trong bộ nhớ phiên notebook."""

    def __init__(self, hoi_vien):
        self.du_lieu = {
            "hoi_vien": deepcopy(hoi_vien),
            "giao_dich": [],
            "doi_diem": [],
        }
        self._lich_su = []
        self._lap_chi_muc()

    def _lap_chi_muc(self):
        self.hv_theo_ma = {hv["ma_tv"]: hv for hv in self.du_lieu["hoi_vien"]}
        self.gd_theo_ma = {gd["ma_gd"]: gd for gd in self.du_lieu["giao_dich"]}

    def _ghi_nhan(self, trang_thai_moi):
        self._lich_su.append(self.du_lieu)
        self.du_lieu = trang_thai_moi
        self._lap_chi_muc()

    def _kiem_tra_hoi_vien(self, ma_tv):
        ma_tv = ma_tv.strip().upper()
        if ma_tv not in self.hv_theo_ma:
            raise LoiNghiepVu("Không tìm thấy hội viên.")
        return ma_tv

    def _diem_kha_dung(self, ma_tv, du_lieu=None):
        ds = du_lieu if du_lieu is not None else self.du_lieu
        diem_cong = sum(
            gd["diem"] for gd in ds["giao_dich"]
            if gd["ma_tv"] == ma_tv and gd["trang_thai"] == "da_duyet" and not gd["da_hoan"]
        )
        diem_doi = sum(
            gd["diem"] for gd in ds["doi_diem"] if gd["ma_tv"] == ma_tv
        )
        return diem_cong - diem_doi

    def them_giao_dich(self, ma_tv, so_tien_vnd, danh_muc):
        ma_tv = self._kiem_tra_hoi_vien(ma_tv)
        if isinstance(so_tien_vnd, bool) or not isinstance(so_tien_vnd, int):
            raise LoiNghiepVu("Số tiền phải là số nguyên VND.")
        if not 1_000 <= so_tien_vnd <= 1_000_000_000:
            raise LoiNghiepVu("Số tiền phải từ 1.000 đến 1.000.000.000 VND.")
        danh_muc = danh_muc.strip()
        if not danh_muc or len(danh_muc) > 40:
            raise LoiNghiepVu("Danh mục giao dịch không hợp lệ.")
        so_ma = [
            int(gd["ma_gd"][2:]) for gd in self.du_lieu["giao_dich"]
            if gd["ma_gd"].startswith("GD") and gd["ma_gd"][2:].isdigit()
        ]
        gd = {
            "ma_gd": f"GD{max(so_ma, default=0) + 1:04d}",
            "ma_tv": ma_tv,
            "so_tien_vnd": so_tien_vnd,
            "danh_muc": danh_muc,
            "hoan_tien_vnd": so_tien_vnd // 100,
            "diem": so_tien_vnd // VND_MOI_DIEM,
            "trang_thai": "cho_duyet" if so_tien_vnd > NGUONG_DUYET_VND else "da_duyet",
            "da_hoan": False,
        }
        moi = deepcopy(self.du_lieu)
        moi["giao_dich"].append(gd)
        self._ghi_nhan(moi)
        return deepcopy(gd)

    def tim_giao_dich(self, ma_gd):
        gd = self.gd_theo_ma.get(ma_gd.strip().upper())
        return deepcopy(gd) if gd is not None else None

    def hang_cho_fifo(self):
        return deque(
            gd["ma_gd"] for gd in self.du_lieu["giao_dich"]
            if gd["trang_thai"] == "cho_duyet"
        )

    def hang_cho_uu_tien(self):
        heap = [
            (-gd["so_tien_vnd"], gd["ma_gd"])
            for gd in self.du_lieu["giao_dich"] if gd["trang_thai"] == "cho_duyet"
        ]
        heapq.heapify(heap)
        return [self.gd_theo_ma[heapq.heappop(heap)[1]] for _ in range(len(heap))]

    def duyet_tiep(self, chien_luoc="uu_tien"):
        if chien_luoc not in ("uu_tien", "fifo"):
            raise LoiNghiepVu("Chiến lược phải là uu_tien hoặc fifo.")
        cho_duyet = self.hang_cho_fifo()
        if not cho_duyet:
            return None
        if chien_luoc == "fifo":
            ma_gd = cho_duyet.popleft()
        else:
            heap = [
                (-gd["so_tien_vnd"], gd["ma_gd"])
                for gd in self.du_lieu["giao_dich"] if gd["trang_thai"] == "cho_duyet"
            ]
            heapq.heapify(heap)
            ma_gd = heapq.heappop(heap)[1]
        moi = deepcopy(self.du_lieu)
        next(gd for gd in moi["giao_dich"] if gd["ma_gd"] == ma_gd)["trang_thai"] = "da_duyet"
        self._ghi_nhan(moi)
        return self.tim_giao_dich(ma_gd)

    def doi_diem(self, ma_tv, diem):
        ma_tv = self._kiem_tra_hoi_vien(ma_tv)
        if isinstance(diem, bool) or not isinstance(diem, int):
            raise LoiNghiepVu("Điểm đổi phải là số nguyên.")
        if diem < BUOC_DOI_DIEM or diem % BUOC_DOI_DIEM:
            raise LoiNghiepVu("Điểm đổi phải từ 100 và chia hết cho 100.")
        if diem > self._diem_kha_dung(ma_tv):
            raise LoiNghiepVu("Số điểm khả dụng không đủ.")
        moi = deepcopy(self.du_lieu)
        phieu = {
            "ma_doi": f"HT{len(moi['doi_diem']) + 1:04d}",
            "ma_tv": ma_tv,
            "diem": diem,
            "gia_tri_vnd": diem * VND_DOI_MOI_DIEM,
        }
        moi["doi_diem"].append(phieu)
        self._ghi_nhan(moi)
        return deepcopy(phieu)

    def hoan_giao_dich(self, ma_gd):
        ma_gd = ma_gd.strip().upper()
        gd = self.gd_theo_ma.get(ma_gd)
        if gd is None:
            raise LoiNghiepVu("Không tìm thấy giao dịch.")
        if gd["trang_thai"] != "da_duyet" or gd["da_hoan"]:
            raise LoiNghiepVu("Chỉ hoàn giao dịch đã duyệt và chưa hoàn.")
        if self._diem_kha_dung(gd["ma_tv"]) - gd["diem"] < 0:
            raise LoiNghiepVu("Không thể hoàn: điểm từ giao dịch đã được sử dụng.")
        moi = deepcopy(self.du_lieu)
        next(item for item in moi["giao_dich"] if item["ma_gd"] == ma_gd)["da_hoan"] = True
        self._ghi_nhan(moi)
        return self.tim_giao_dich(ma_gd)

    def hoan_tac(self):
        if not self._lich_su:
            return False
        self.du_lieu = self._lich_su.pop()
        self._lap_chi_muc()
        return True

    def tong_quan(self):
        hoi_vien = []
        for hv in self.du_lieu["hoi_vien"]:
            gd_hv = [
                gd for gd in self.du_lieu["giao_dich"]
                if gd["ma_tv"] == hv["ma_tv"] and gd["trang_thai"] == "da_duyet" and not gd["da_hoan"]
            ]
            hoi_vien.append({
                **hv,
                "diem_kha_dung": self._diem_kha_dung(hv["ma_tv"]),
                "hoan_tien_vnd": sum(gd["hoan_tien_vnd"] for gd in gd_hv),
            })
        gd_duyet = [
            gd for gd in self.du_lieu["giao_dich"]
            if gd["trang_thai"] == "da_duyet" and not gd["da_hoan"]
        ]
        return {
            "hoi_vien": hoi_vien,
            "giao_dich": merge_sort_giao_dich(self.du_lieu["giao_dich"]),
            "tong_giao_dich": len(self.du_lieu["giao_dich"]),
            "cho_duyet": sum(gd["trang_thai"] == "cho_duyet" for gd in self.du_lieu["giao_dich"]),
            "tong_hoan_tien_vnd": sum(gd["hoan_tien_vnd"] for gd in gd_duyet),
            "tong_diem_kha_dung": sum(hv["diem_kha_dung"] for hv in hoi_vien),
        }

def tao_du_lieu_demo():
    hoi_vien = [
        {"ma_tv": "TV001", "ten": "Nguyễn An", "hang": "Bạc"},
        {"ma_tv": "TV002", "ten": "Trần Bình", "hang": "Vàng"},
        {"ma_tv": "TV003", "ten": "Lê Chi", "hang": "Bạc"},
        {"ma_tv": "TV004", "ten": "Phạm Duy", "hang": "Thành viên"},
        {"ma_tv": "TV005", "ten": "Võ Hà", "hang": "Vàng"},
    ]
    book = SoThuong(hoi_vien)
    mau = [
        ("TV001", 1_250_000, "Siêu thị"), ("TV002", 735_000, "Ăn uống"),
        ("TV003", 4_900_000, "Du lịch"), ("TV004", 6_250_000, "Điện tử"),
        ("TV005", 2_180_000, "Siêu thị"), ("TV001", 500_000, "Di chuyển"),
        ("TV002", 9_900_000, "Nội thất"), ("TV003", 1_380_000, "Nhà thuốc"),
    ]
    for ma_tv, so_tien, danh_muc in mau:
        book.them_giao_dich(ma_tv, so_tien, danh_muc)
    return book

book = tao_du_lieu_demo()
print("Đã khởi tạo sổ giao dịch giả lập và các chỉ mục tra cứu.")

## 3. Dashboard và sắp xếp

Merge sort chạy O(n log n), dùng mã giao dịch để xử lý trường hợp bằng số tiền. Dashboard chỉ cộng quyền lợi của giao dịch đã duyệt và chưa hoàn.

In [ ]:
def dinh_dang_vnd(so_tien):
    return f"{so_tien:,}".replace(",", ".") + " ₫"

tong = book.tong_quan()
print("TỔNG QUAN HỘI VIÊN VÀ GIAO DỊCH")
print(f"Giao dịch: {tong['tong_giao_dich']} | Chờ duyệt: {tong['cho_duyet']} | "
      f"Hoàn tiền: {dinh_dang_vnd(tong['tong_hoan_tien_vnd'])} | "
      f"Điểm khả dụng: {tong['tong_diem_kha_dung']:,}")
print("\nMÃ GD   | HỘI VIÊN | SỐ TIỀN        | HOÀN TIỀN    | ĐIỂM | TRẠNG THÁI")
print("-" * 78)
for gd in tong["giao_dich"]:
    ten = book.hv_theo_ma[gd["ma_tv"]]["ten"]
    trang_thai = "Đã hoàn" if gd["da_hoan"] else gd["trang_thai"]
    thuong = "—" if gd["trang_thai"] == "cho_duyet" or gd["da_hoan"] else dinh_dang_vnd(gd["hoan_tien_vnd"])
    diem = "—" if gd["trang_thai"] == "cho_duyet" or gd["da_hoan"] else gd["diem"]
    print(f"{gd['ma_gd']} | {ten:<9} | {dinh_dang_vnd(gd['so_tien_vnd']):>13} | "
          f"{thuong:>12} | {str(diem):>4} | {trang_thai}")

print("\nHội viên xếp theo điểm khả dụng:")
for hv in sorted(tong["hoi_vien"], key=lambda x: (-x["diem_kha_dung"], x["ma_tv"])):
    print(f"{hv['ma_tv']} · {hv['ten']}: {hv['diem_kha_dung']} điểm | "
          f"hoàn {dinh_dang_vnd(hv['hoan_tien_vnd'])}")

## 4. Tra cứu và hàng chờ ưu tiên

Tra cứu mã dùng bảng băm trung bình O(1). Hàng đợi FIFO lấy giao dịch đến trước; heap ưu tiên khoản có số tiền cao nhất. Hai danh sách dưới đây được xem trước, chưa làm thay đổi trạng thái sổ.

In [ ]:
ket_qua = book.tim_giao_dich("gd0001")
print("Tra cứu GD0001:", ket_qua)
print("Tra cứu không tồn tại:", book.tim_giao_dich("GD9999"))
print("\nThứ tự FIFO:", list(book.hang_cho_fifo()))
print("Ưu tiên số tiền cao nhất:")
for gd in book.hang_cho_uu_tien():
    print(f"{gd['ma_gd']} · {dinh_dang_vnd(gd['so_tien_vnd'])}")
print("\nSắp xếp số tiền thấp → cao:")
for gd in merge_sort_giao_dich(book.du_lieu["giao_dich"], "asc"):
    print(gd["ma_gd"], dinh_dang_vnd(gd["so_tien_vnd"]))

## 5. Thao tác nghiệp vụ

Cell dưới đây chạy thử các luồng thêm giao dịch thông thường, giao dịch chờ duyệt, duyệt theo heap, đổi điểm, kiểm tra hoàn giao dịch, rồi hoàn tác thao tác đổi điểm. Cell tạo lại sổ demo trước khi chạy để có thể chạy lại độc lập.

In [ ]:
book = tao_du_lieu_demo()
giao_dich_moi = book.them_giao_dich("TV001", 850_000, "Siêu thị")
print("Giao dịch mới:", giao_dich_moi)
giao_dich_lon = book.them_giao_dich("TV004", 9_000_000, "Điện tử")
print("Khoản lớn chuyển chờ duyệt:", giao_dich_lon["ma_gd"], giao_dich_lon["trang_thai"])
da_duyet = book.duyet_tiep("uu_tien")
print("Duyệt ưu tiên khoản lớn nhất:", da_duyet["ma_gd"], dinh_dang_vnd(da_duyet["so_tien_vnd"]))
phieu = book.doi_diem("TV003", 100)
print("Đổi điểm:", phieu)
try:
    book.hoan_giao_dich("GD0001")
except LoiNghiepVu as loi:
    print("Kiểm tra hoàn giao dịch đã dùng điểm:", loi)
print("Hoàn tác đổi điểm:", book.hoan_tac())
print("Số dư TV003 sau hoàn tác:", book._diem_kha_dung("TV003"), "điểm")

## 6. Kiểm thử tự động

Các kiểm tra dưới đây xác nhận tìm kiếm, merge sort, điều kiện duyệt, FIFO/ưu tiên, đổi/hoàn điểm, undo và từ chối dữ liệu sai. Mỗi test dùng sổ riêng để không phụ thuộc trạng thái demo.

In [ ]:
def kiem_thu_notebook():
    so_test = 0
    def dat_dieu_kien(dieu_kien, ten):
        nonlocal so_test
        assert dieu_kien, f"Không đạt: {ten}"
        so_test += 1
        print(f"✓ {ten}")

    s = tao_du_lieu_demo()
    dat_dieu_kien(s.tim_giao_dich("gd0001")["ma_tv"] == "TV001", "Tra cứu mã không phân biệt hoa thường")
    dat_dieu_kien(s.tim_giao_dich("GD9999") is None, "Mã không tồn tại trả None")
    ds = [{"ma_gd": "GD02", "so_tien_vnd": 10}, {"ma_gd": "GD01", "so_tien_vnd": 10}]
    dat_dieu_kien(merge_sort_giao_dich([]) == [], "Sắp xếp danh sách rỗng")
    dat_dieu_kien([x["ma_gd"] for x in merge_sort_giao_dich(ds, "asc")] == ["GD01", "GD02"], "Merge sort tăng và phá hòa theo mã")
    dat_dieu_kien(s.hang_cho_fifo()[0] == "GD0004", "FIFO giữ thứ tự đến")
    dat_dieu_kien(s.duyet_tiep("uu_tien")["ma_gd"] == "GD0007", "Heap lấy giao dịch giá trị cao nhất")

    diem_truoc = s._diem_kha_dung("TV001")
    try:
        s.doi_diem("TV001", 200)
        doi_thieu = False
    except LoiNghiepVu:
        doi_thieu = True
    dat_dieu_kien(doi_thieu and s._diem_kha_dung("TV001") == diem_truoc, "Từ chối đổi vượt số dư, không thay đổi dữ liệu")

    s.doi_diem("TV003", 200)
    try:
        s.hoan_giao_dich("GD0003")
        hoan_bi_chan = False
    except LoiNghiepVu:
        hoan_bi_chan = True
    dat_dieu_kien(hoan_bi_chan, "Chặn hoàn giao dịch khi điểm đã dùng")
    dat_dieu_kien(s.hoan_tac() and s._diem_kha_dung("TV003") == 628, "Undo khôi phục giao dịch đổi điểm")

    thu = SoThuong([{"ma_tv": "TV001", "ten": "Test", "hang": "Bạc"}])
    binh_thuong = thu.them_giao_dich("TV001", 1_000_000, "Siêu thị")
    cho = thu.them_giao_dich("TV001", 5_000_001, "Điện tử")
    dat_dieu_kien(binh_thuong["hoan_tien_vnd"] == 10_000 and binh_thuong["diem"] == 100, "Tính tiền hoàn và điểm")
    dat_dieu_kien(cho["trang_thai"] == "cho_duyet", "Giao dịch vượt ngưỡng cần duyệt")
    hoan = thu.hoan_giao_dich(binh_thuong["ma_gd"])
    dat_dieu_kien(hoan["da_hoan"] and thu._diem_kha_dung("TV001") == 0, "Hoàn giao dịch thu hồi điểm chưa dùng")
    try:
        thu.them_giao_dich("TV999", 100_000, "Siêu thị")
        du_lieu_sai = False
    except LoiNghiepVu:
        du_lieu_sai = True
    dat_dieu_kien(du_lieu_sai and thu.tong_quan()["tong_giao_dich"] == 2, "Từ chối hội viên không tồn tại")
    dat_dieu_kien(thu.hang_cho_fifo() == deque(["GD0002"]), "Hàng chờ sau khi hoàn giao dịch")
    return so_test

so_test_dat = kiem_thu_notebook()
print(f"\nKẾT QUẢ: {so_test_dat}/14 kiểm tra đạt.")

## 7. Thực nghiệm hiệu năng

So sánh merge sort với `sorted` (Timsort của Python) và quét tuyến tính với dict. Seed cố định giúp tái tạo bộ dữ liệu; thời gian cụ thể phụ thuộc máy/Colab và không đại diện tải production.

In [ ]:
def thoi_gian_tb_ms(thao_tac, so_lan=5):
    mau_do = []
    for _ in range(so_lan):
        bat_dau = perf_counter()
        thao_tac()
        mau_do.append((perf_counter() - bat_dau) * 1_000)
    return fmean(mau_do)

rng = random.Random(20261013)
print("N      | Merge sort ms | sorted ms | Quét tuần tự µs | Dict µs")
print("-" * 70)
for n in (100, 1_000, 10_000):
    ds = [{"ma_gd": f"GD{i:05d}", "so_tien_vnd": rng.randrange(1_000, 10**9)} for i in range(n)]
    rng.shuffle(ds)
    chi_muc = {gd["ma_gd"]: gd for gd in ds}
    ma_can_tim = ds[n // 2]["ma_gd"]
    so_lan = 3
    merge_ms = thoi_gian_tb_ms(lambda: merge_sort_giao_dich(ds, "asc"), so_lan)
    sorted_ms = thoi_gian_tb_ms(lambda: sorted(ds, key=lambda gd: gd["so_tien_vnd"]), so_lan)
    tuyen_tinh_us = thoi_gian_tb_ms(lambda: next(gd for gd in ds if gd["ma_gd"] == ma_can_tim), so_lan) * 1_000
    dict_us = thoi_gian_tb_ms(lambda: chi_muc[ma_can_tim], so_lan) * 1_000
    print(f"{n:<6} | {merge_ms:>12.4f} | {sorted_ms:>9.4f} | {tuyen_tinh_us:>15.4f} | {dict_us:>7.4f}")

## 8. Kết luận và giới hạn

- Notebook thực hiện được CRUD tối thiểu cho giao dịch, tra cứu, sắp xếp, ưu tiên/FIFO, đổi/hoàn điểm và kiểm thử ngay trên Google Colab/Jupyter.
- List, Queue, Stack, Hash Table và Heap được áp dụng vào đúng thao tác nghiệp vụ; merge sort là thuật toán sắp xếp trên List. Test xác nhận các trường hợp chính và biên.
- Trạng thái notebook chỉ sống trong RAM; reset hoặc khởi động lại kernel sẽ xóa phiên làm việc. Notebook không có xác thực, cơ sở dữ liệu, audit trail hoặc kết nối ngân hàng.
- Các chính sách hoàn/điểm chỉ là dữ liệu minh họa, không áp dụng cho sản phẩm tài chính thật.

**Nộp bài:** nộp tệp notebook này cùng mã nguồn ứng dụng và báo cáo; cập nhật tên thành viên, mức đóng góp và chèn kết quả benchmark thực tế chạy trên môi trường của nhóm.